# Glacier Lake Watch v2: run everything

**Click Runtime → Run all.** That's it.

- When Google asks to connect to Google Drive, click **Connect** and allow it. Results get saved to the `Glacier Lake Watch` folder in your Drive.
- The first run downloads about 100 small satellite images (20 to 40 minutes). If Colab disconnects, just click Run all again: downloads are saved in Drive and it picks up where it stopped.
- No GPU needed.

In [ ]:
# 1. Unpack the project code (built into this notebook)
import base64, io, tarfile, os, shutil
CODE = '...'
shutil.rmtree('/content/glw', ignore_errors=True)
os.makedirs('/content/glw', exist_ok=True)
tarfile.open(fileobj=io.BytesIO(base64.b64decode(CODE)), mode='r:gz').extractall('/content/glw')
%cd /content/glw
print(sorted(os.listdir('.')))

In [ ]:
# 2. Install the few libraries Colab doesn't already have
!pip install -q rasterio pystac-client gradio plotly shapely

In [ ]:
# 3. Connect Google Drive so downloads and results are saved
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/Glacier Lake Watch'
os.makedirs(BASE + '/cache_v2', exist_ok=True)
os.makedirs('data', exist_ok=True)
if not os.path.islink('data/chips'):
    os.symlink(BASE + '/cache_v2', 'data/chips')
print('Saving to', BASE)

In [ ]:
# 4. Download data, train, test on unseen lakes, measure, forecast, build report card
!python run_all.py --workers 16

In [ ]:
# 5. Save results to Google Drive
import shutil
shutil.copytree('results', BASE + '/results_v2', dirs_exist_ok=True)
shutil.make_archive(BASE + '/results_v2', 'zip', 'results')
print('Saved results to', BASE + '/results_v2')

In [ ]:
# 6. Show the results
from IPython.display import Markdown, Image, display
import glob
display(Markdown(open('results/RESULTS.md').read().split('![')[0]))
for f in sorted(glob.glob('results/figures/*.png')):
    print(f); display(Image(f))

## Optional: open the web app
Run the cell below for a temporary public link to the web app.

In [ ]:
import sys
sys.argv = ['app.py', '--share']
exec(open('app.py').read())